In [5]:
def dls_solve_8_puzzle(initial_board, goal_board, limit):
    initial_state = PuzzleState(initial_board)
    goal_state = PuzzleState(goal_board)

    # Stack stores tuples of (PuzzleState, current_depth)
    stack = [(initial_state, 0)]
    visited = set()
    visited.add(initial_state)

    while stack:
        current_state, current_depth = stack.pop()

        if current_state == goal_state:
            path = []
            temp_state = current_state
            while temp_state:
                path.append(temp_state)
                temp_state = temp_state.parent
            return path[::-1]  # Return the path from initial to goal

        # Only explore successors if within the depth limit
        if current_depth < limit:
            # Get successors in reverse order to explore 'leftmost' first (DFS behavior with pop())
            successors = current_state.get_successors()
            for successor in reversed(successors):
                if successor not in visited:
                    visited.add(successor)
                    stack.append((successor, current_depth + 1))

    return None  # No solution found within the given depth limit

def get_board_input(prompt_message):
    board = []
    print(prompt_message)
    print("Enter 3 rows of 3 numbers (0-8) separated by spaces. Use 0 for the blank space.")
    for i in range(3):
        while True:
            try:
                row_str = input(f"Enter row {i+1}: ")
                row = [int(x) for x in row_str.split()]
                if len(row) == 3 and all(0 <= x <= 8 for x in row):
                    board.append(row)
                    break
                else:
                    print("Invalid input. Please enter exactly 3 numbers between 0 and 8.")
            except ValueError:
                print("Invalid input. Please enter numbers separated by spaces.")
    return board

# Get initial and goal board from user
user_initial_board = get_board_input("--- Initial Board Setup ---")
user_goal_board = get_board_input("--- Goal Board Setup ---")

# Get depth limit from user
while True:
    try:
        depth_limit = int(input("Enter the depth limit for DLS: "))
        if depth_limit > 0:
            break
        else:
            print("Depth limit must be a positive integer.")
    except ValueError:
        print("Invalid input. Please enter an integer.")

print("\nInitial Board:")
PuzzleState(user_initial_board).print_board()
print("Goal Board:")
PuzzleState(user_goal_board).print_board()
print(f"Searching with Depth Limit: {depth_limit}")

solution_path_dls = dls_solve_8_puzzle(user_initial_board, user_goal_board, depth_limit)

if solution_path_dls:
    print(f"\nSolution found within depth limit {depth_limit} in {len(solution_path_dls) - 1} steps:")
    for i, state in enumerate(solution_path_dls):
        print(f"Step {i}: {state.move if state.move else 'Start'}")
        state.print_board()
else:
    print(f"\nNo solution found within the depth limit of {depth_limit}.")
    print("You might need to increase the depth limit or the puzzle is unsolvable.")

--- Initial Board Setup ---
Enter 3 rows of 3 numbers (0-8) separated by spaces. Use 0 for the blank space.
Enter row 1: 2 8 3
Enter row 2: 1 6 4
Enter row 3: 7 0 5
--- Goal Board Setup ---
Enter 3 rows of 3 numbers (0-8) separated by spaces. Use 0 for the blank space.
Enter row 1: 1 2 3 
Enter row 2: 8 0 4
Enter row 3: 7 6  5
Enter the depth limit for DLS: 5

Initial Board:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

Goal Board:
[1, 2, 3]
[8, 0, 4]
[7, 6, 5]

Searching with Depth Limit: 5

Solution found within depth limit 5 in 5 steps:
Step 0: Start
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

Step 1: Up
[2, 8, 3]
[1, 0, 4]
[7, 6, 5]

Step 2: Up
[2, 0, 3]
[1, 8, 4]
[7, 6, 5]

Step 3: Left
[0, 2, 3]
[1, 8, 4]
[7, 6, 5]

Step 4: Down
[1, 2, 3]
[0, 8, 4]
[7, 6, 5]

Step 5: Right
[1, 2, 3]
[8, 0, 4]
[7, 6, 5]



In [6]:
class PuzzleState:
    def __init__(self, board, parent=None, move=None):
        self.board = board
        self.parent = parent
        self.move = move

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(str(self.board))

    def get_blank_position(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c
        return -1, -1 # Should not happen in a valid 8-puzzle

    def get_successors(self):
        successors = []
        r_blank, c_blank = self.get_blank_position()

        # Possible moves: Up, Down, Left, Right
        moves = [
            (-1, 0, 'Up'),    # Up
            (1, 0, 'Down'),   # Down
            (0, -1, 'Left'),  # Left
            (0, 1, 'Right')   # Right
        ]

        for dr, dc, move_name in moves:
            new_r, new_c = r_blank + dr, c_blank + dc

            if 0 <= new_r < 3 and 0 <= new_c < 3:
                new_board = [row[:] for row in self.board]
                new_board[r_blank][c_blank], new_board[new_r][new_c] = new_board[new_r][new_c], new_board[r_blank][c_blank]
                successors.append(PuzzleState(new_board, self, move_name))
        return successors

    def print_board(self):
        for row in self.board:
            print(row)
        print()

def dfs_solve_8_puzzle(initial_board, goal_board):
    initial_state = PuzzleState(initial_board)
    goal_state = PuzzleState(goal_board)

    stack = [initial_state]
    visited = set()
    visited.add(initial_state)

    while stack:
        current_state = stack.pop()

        if current_state == goal_state:
            path = []
            temp_state = current_state
            while temp_state:
                path.append(temp_state)
                temp_state = temp_state.parent
            return path[::-1] # Return the path from initial to goal

        for successor in current_state.get_successors():
            if successor not in visited:
                visited.add(successor)
                stack.append(successor)
    return None # No solution found

def get_board_input(prompt_message):
    board = []
    print(prompt_message)
    print("Enter 3 rows of 3 numbers (0-8) separated by spaces. Use 0 for the blank space.")
    for i in range(3):
        while True:
            try:
                row_str = input(f"Enter row {i+1}: ")
                row = [int(x) for x in row_str.split()]
                if len(row) == 3 and all(0 <= x <= 8 for x in row):
                    # Basic check for uniqueness within the row (0 is blank, others should be unique)
                    if len(set(row)) == 3 or (0 in row and len(set(row)) == 2):
                        board.append(row)
                        break
                    else:
                        print("Invalid input. Numbers in a row must be unique (except for 0). Duplicate numbers detected.")
                else:
                    print("Invalid input. Please enter exactly 3 numbers between 0 and 8.")
            except ValueError:
                print("Invalid input. Please enter numbers separated by spaces.")

    # Further validation for the whole board to ensure all numbers 0-8 are present and unique
    flat_board = [num for sublist in board for num in sublist]
    if len(set(flat_board)) != 9:
        print("Warning: The numbers 0-8 are not all present and unique across the entire board. This might lead to an invalid or unsolvable puzzle state.")
    return board

# Get initial and goal board from user
user_initial_board = get_board_input("--- Initial Board Setup ---")
user_goal_board = get_board_input("--- Goal Board Setup ---")

print("\nInitial Board:")
PuzzleState(user_initial_board).print_board()
print("Goal Board:")
PuzzleState(user_goal_board).print_board()

print("Solving using DFS...")
solution_path = dfs_solve_8_puzzle(user_initial_board, user_goal_board)

if solution_path:
    print(f"Solution found in {len(solution_path) - 1} steps:")
    for i, state in enumerate(solution_path):
        print(f"Step {i}: {state.move if state.move else 'Start'}")
        state.print_board()
else:
    print("No solution found for the given initial and goal states using DFS.")
    print("DFS might not find a solution if the state space is too large or if it gets stuck in a deep path.")

Streaming output truncated to the last 5000 lines.
Step 2438: Up
[5, 4, 8]
[0, 1, 7]
[6, 2, 3]

Step 2439: Right
[5, 4, 8]
[1, 0, 7]
[6, 2, 3]

Step 2440: Right
[5, 4, 8]
[1, 7, 0]
[6, 2, 3]

Step 2441: Up
[5, 4, 0]
[1, 7, 8]
[6, 2, 3]

Step 2442: Left
[5, 0, 4]
[1, 7, 8]
[6, 2, 3]

Step 2443: Left
[0, 5, 4]
[1, 7, 8]
[6, 2, 3]

Step 2444: Down
[1, 5, 4]
[0, 7, 8]
[6, 2, 3]

Step 2445: Right
[1, 5, 4]
[7, 0, 8]
[6, 2, 3]

Step 2446: Right
[1, 5, 4]
[7, 8, 0]
[6, 2, 3]

Step 2447: Up
[1, 5, 0]
[7, 8, 4]
[6, 2, 3]

Step 2448: Left
[1, 0, 5]
[7, 8, 4]
[6, 2, 3]

Step 2449: Left
[0, 1, 5]
[7, 8, 4]
[6, 2, 3]

Step 2450: Down
[7, 1, 5]
[0, 8, 4]
[6, 2, 3]

Step 2451: Right
[7, 1, 5]
[8, 0, 4]
[6, 2, 3]

Step 2452: Right
[7, 1, 5]
[8, 4, 0]
[6, 2, 3]

Step 2453: Down
[7, 1, 5]
[8, 4, 3]
[6, 2, 0]

Step 2454: Left
[7, 1, 5]
[8, 4, 3]
[6, 0, 2]

Step 2455: Left
[7, 1, 5]
[8, 4, 3]
[0, 6, 2]

Step 2456: Up
[7, 1, 5]
[0, 4, 3]
[8, 6, 2]

Step 2457: Right
[7, 1, 5]
[4, 0, 3]
[8, 6, 2]

Step 2458: